Libraries import

In [54]:
import pandas as pd


Datasets import

I was able to do some initial data cleaning directly on the Statistics Canada website before downloading the datasets. I      filtered the data to include only 2023 and selected the Canada-wide average rather than individual provinces.x

In [55]:
assets_and_debts = pd.read_csv('/Users/mateonunez/Desktop/Codecademy_Under_30_Consumer_Spending/data/1110001601_databaseLoadingData.csv')
assets_and_debts.head()

,REF_DATE,GEO,DGUID,Assets and debts,Age group,Economic family type,Statistics,Confidence intervals,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2023,Canada,2016A000011124,Total assets,Under 35 years,Economic families and persons not in an econom...,Total values,Estimate,2023 constant dollars,449,millions,6,v80526139,1.1.3.1.1.1,1958404.0,NaN,NaN,NaN,0
1,2023,Canada,2016A000011124,Total assets,Under 35 years,Economic families and persons not in an econom...,Percentage of total assets or total debts,Estimate,Percentage,242,units,0,v80526140,1.1.3.1.2.1,100.0,NaN,NaN,NaN,1
2,2023,Canada,2016A000011124,Total assets,Under 35 years,Economic families and persons not in an econom...,Number holding asset or debt,Estimate,Number,223,thousands,3,v80526141,1.1.3.1.3.1,3232.0,NaN,NaN,NaN,0
3,2023,Canada,2016A000011124,Total assets,Under 35 years,Economic families and persons not in an econom...,Percentage holding asset or debt,Estimate,Percentage,242,units,0,v80526142,1.1.3.1.4.1,100.0,NaN,NaN,NaN,1
4,2023,Canada,2016A000011124,Total assets,Under 35 years,Economic families and persons not in an econom...,Median value for those holding asset or debt,Estimate,2023 constant dollars,449,units,0,v80526143,1.1.3.1.5.1,245200.0,NaN,NaN,NaN,0


In [56]:
household_spending = pd.read_csv('/Users/mateonunez/Desktop/Codecademy_Under_30_Consumer_Spending/data/1110022701_databaseLoadingData.csv')
household_spending.head()

,REF_DATE,GEO,DGUID,Statistic,Age of reference person,"Household expenditures, summary-level categories",UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2023,Canada,2016A000011124,Average expenditure per household,Less than 30 years,Total expenditure,Dollars,81,units,0,v54531042,1.1.2.1,86051.0,NaN,NaN,NaN,0
1,2023,Canada,2016A000011124,Average expenditure per household,Less than 30 years,Total current consumption,Dollars,81,units,0,v54531043,1.1.2.2,68299.0,NaN,NaN,NaN,0
2,2023,Canada,2016A000011124,Average expenditure per household,Less than 30 years,Food purchased from stores,Dollars,81,units,0,v54531045,1.1.2.4,5684.0,NaN,NaN,NaN,0
3,2023,Canada,2016A000011124,Average expenditure per household,Less than 30 years,Food purchased from restaurants,Dollars,81,units,0,v54531046,1.1.2.5,3973.0,NaN,NaN,NaN,0
4,2023,Canada,2016A000011124,Average expenditure per household,Less than 30 years,Principal accommodation,Dollars,81,units,0,v54531048,1.1.2.7,22505.0,NaN,NaN,NaN,0


## Spending comparison



In [57]:
spending = household_spending[
    (household_spending['Age of reference person'] == 'Less than 30 years') |
    (household_spending['Age of reference person'] == '30 to 39 years')
].rename(columns={
    'Age of reference person': 'Age',
    'Household expenditures, summary-level categories': 'Category'
}).reset_index(drop=True)

spending['Group'] = None
spending.loc[spending['Category'] == 'Principal accommodation', 'Group'] = 'Housing'
spending.loc[(spending['Category'] == 'Food purchased from stores') |
             (spending['Category'] == 'Food purchased from restaurants'), 'Group'] = 'Food'
spending.loc[(spending['Category'] == 'Private transportation') |
             (spending['Category'] == 'Public transportation'), 'Group'] = 'Transportation'
spending.loc[spending['Category'] == 'Personal insurance payments and pension contributions', 'Group'] = 'Insurance and pension contributions'

young_total = spending.loc[(spending['Age'] == 'Less than 30 years') &
                           (spending['Category'] == 'Total expenditure'), 'VALUE'].iloc[0]
next_total = spending.loc[(spending['Age'] == '30 to 39 years') &
                          (spending['Category'] == 'Total expenditure'), 'VALUE'].iloc[0]
spending_summary = spending.groupby(['Age', 'Group']).agg({'VALUE': 'sum'}).reset_index()
spending_summary.loc[spending_summary['Age'] == 'Less than 30 years', 'Budget share (%)'] = spending_summary['VALUE'] / young_total * 100
spending_summary.loc[spending_summary['Age'] == '30 to 39 years', 'Budget share (%)'] = spending_summary['VALUE'] / next_total * 100
spending_comparison = spending_summary.pivot_table(index='Group', columns='Age', values='Budget share (%)')
spending_comparison['Difference (percentage points)'] = spending_comparison['Less than 30 years'] - spending_comparison['30 to 39 years']
spending_comparison.round(1)

Age,30 to 39 years,Less than 30 years,Difference (percentage points)
Group,,,
Food,10.6,11.2,0.6
Housing,24.1,26.2,2.1
Insurance and pension contributions,7.1,6.7,-0.4
Transportation,10.9,12.7,1.8


## Balance sheet comparison



In [68]:
balance_sheet = assets_and_debts.loc[
    ((assets_and_debts['Age group'] == 'Under 35 years') |
     (assets_and_debts['Age group'] == '35 to 44 years')) &
    (assets_and_debts['Economic family type'] == 'Economic families and persons not in an economic family') &
    ((assets_and_debts['Statistics'] == 'Percentage holding asset or debt') |
     (assets_and_debts['Statistics'] == 'Median value for those holding asset or debt')),
    ['Age group', 'Assets and debts', 'Statistics', 'VALUE', 'UOM', 'STATUS']
].sort_values(by=['Assets and debts', 'Statistics', 'Age group']).reset_index(drop=True)
display(balance_sheet)

,Age group,Assets and debts,Statistics,VALUE,UOM,STATUS
0,35 to 44 years,Bonds (saving and other),Median value for those holding asset or debt,NaN,2023 constant dollars,F
1,Under 35 years,Bonds (saving and other),Median value for those holding asset or debt,NaN,2023 constant dollars,F
2,35 to 44 years,Bonds (saving and other),Percentage holding asset or debt,NaN,Percentage,F
3,Under 35 years,Bonds (saving and other),Percentage holding asset or debt,NaN,Percentage,F
4,35 to 44 years,Credit card and installment debt,Median value for those holding asset or debt,5000.0,2023 constant dollars,NaN
...,...,...,...,...,...,...
115,Under 35 years,Vehicles,Percentage holding asset or debt,73.2,Percentage,NaN
116,35 to 44 years,Vehicles and other non-financial assets,Median value for those holding asset or debt,44000.0,2023 constant dollars,NaN
117,Under 35 years,Vehicles and other non-financial assets,Median value for those holding asset or debt,31000.0,2023 constant dollars,NaN
118,35 to 44 years,Vehicles and other non-financial assets,Percentage holding asset or debt,100.0,Percentage,NaN


In [72]:
have_acar = balance_sheet['Assets and debts'] == 'Vehicles'
have_acarloan = balance_sheet['Assets and debts'] == 'Vehicle loans'
vehicles_and_loans = balance_sheet[have_acar | have_acarloan]
display(vehicles_and_loans)

,Age group,Assets and debts,Statistics,VALUE,UOM,STATUS
108,35 to 44 years,Vehicle loans,Median value for those holding asset or debt,21000.0,2023 constant dollars,NaN
109,Under 35 years,Vehicle loans,Median value for those holding asset or debt,23000.0,2023 constant dollars,NaN
110,35 to 44 years,Vehicle loans,Percentage holding asset or debt,34.7,Percentage,NaN
111,Under 35 years,Vehicle loans,Percentage holding asset or debt,35.3,Percentage,NaN
112,35 to 44 years,Vehicles,Median value for those holding asset or debt,24000.0,2023 constant dollars,NaN
113,Under 35 years,Vehicles,Median value for those holding asset or debt,20000.0,2023 constant dollars,NaN
114,35 to 44 years,Vehicles,Percentage holding asset or debt,80.8,Percentage,NaN
115,Under 35 years,Vehicles,Percentage holding asset or debt,73.2,Percentage,NaN


## Balance sheet pivot tables



In [65]:
under35_pivot = balance_sheet[balance_sheet['Age group'] == 'Under 35 years'].pivot_table(index='Assets and debts', columns='Statistics', values='VALUE', aggfunc='mean', dropna=False)
under35_pivot

Statistics,Median value for those holding asset or debt,Percentage holding asset or debt
Assets and debts,,
Bonds (saving and other),NaN,NaN
Credit card and installment debt,4000.0,42.1
Deposits in financial institutions,6000.0,95.6
Employer-sponsored Registered Pension Plans (EPPs),29700.0,44.4
Equity in business,9100.0,20.0
"Financial assets, non pension",15700.0,97.2
Line of credit,15000.0,17.0
Mortgage debt,232600.0,38.1
Mortgage on other real estate,230000.0,5.4


In [66]:
age35to44_pivot = balance_sheet[balance_sheet['Age group'] == '35 to 44 years'].pivot_table(index='Assets and debts', columns='Statistics', values='VALUE', aggfunc='mean', dropna=False)
age35to44_pivot

Statistics,Median value for those holding asset or debt,Percentage holding asset or debt
Assets and debts,,
Bonds (saving and other),NaN,NaN
Credit card and installment debt,5000.0,44.8
Deposits in financial institutions,6800.0,93.4
Employer-sponsored Registered Pension Plans (EPPs),94800.0,53.5
Equity in business,9700.0,26.5
"Financial assets, non pension",21000.0,96.5
Line of credit,19000.0,22.3
Mortgage debt,247000.0,58.5
Mortgage on other real estate,222000.0,9.8


In [67]:
balance_groups = balance_sheet[
    (balance_sheet['Assets and debts'] == 'Non-financial assets') |
    (balance_sheet['Assets and debts'] == 'Financial assets, non pension') |
    (balance_sheet['Assets and debts'] == 'Private pension assets') |
    (balance_sheet['Assets and debts'] == 'Total debt')
].reset_index(drop=True)
balance_groups['Group'] = 'Investments'
balance_groups.loc[balance_groups['Assets and debts'] == 'Non-financial assets', 'Group'] = 'Assets'
balance_groups.loc[balance_groups['Assets and debts'] == 'Total debt', 'Group'] = 'Debt'
balance_groups = balance_groups.pivot_table(index=['Group', 'Assets and debts', 'Age group'], columns='Statistics', values='VALUE', aggfunc='mean')
balance_groups

Statistics                                                Median value for those holding asset or debt  \
Group       Assets and debts              Age group                                                      
Assets      Non-financial assets          35 to 44 years                                      450000.0   
                                          Under 35 years                                      104000.0   
Debt        Total debt                    35 to 44 years                                      190000.0   
                                          Under 35 years                                       80000.0   
Investments Financial assets, non pension 35 to 44 years                                       21000.0   
                                          Under 35 years                                       15700.0   
            Private pension assets        35 to 44 years                                      100000.0   
                                          Under 35 years                                       35000.0   

Statistics                                                Percentage holding asset or debt  
Group       Assets and debts              Age group                                         
Assets      Non-financial assets          35 to 44 years                             100.0  
                                          Under 35 years                             100.0  
Debt        Total debt                    35 to 44 years                              84.1  
                                          Under 35 years                              78.3  
Investments Financial assets, non pension 35 to 44 years                              96.5  
                                          Under 35 years                              97.2  
            Private pension assets        35 to 44 years                              77.6  
                                          Under 35 years                              67.9